In [7]:
building = 0
time = 24
price = 'Realistic'
noise = 0
full_mpc = False

# Setup

In [3]:
import torch
import pickle
import numpy as np

import src.data.dataprep as prep
import src.data.featurisation as features

import src.optimization.pv_battery as pvb

def torch_py(torch_tensor):
    return torch_tensor.cpu().detach().numpy().flatten()

def rescale(values, scaler):
    rescaled_values = values * (scaler[1] - scaler[0]) + scaler[0]   
    return rescaled_values

def moving_average(data, window_size):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

In [27]:
# Import the base data and resample it from 5 minutes to hourly
nl_data = prep.dutch_data('../data/Dutchdata_clean/building_' + str(building) + '.parquet', 'h', price=price)
featurisation = features.Featurisation(nl_data)
nl_data = featurisation.cyclic_features(yearly=False)[0]

In [28]:
if noise == 0:
    nl_data['load_fcst'] = nl_data['load']
else:
    load_fcst_data = np.load('../data/load_fcsts/building_' + str(building) + '_' + str(noise) + '_noise_forecast.npy')
    nl_data['load_fcst'] = load_fcst_data

In [29]:
nl_data['direct_rad'] = np.load("../../data/DHI_fcsts/DHI_fcst.npy")

In [30]:
battery_capacity = round(nl_data['load'].resample('YE').sum().max() * 1.1 / 1000,1)

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Parameters

In [32]:
# Base parameters
max_charge = battery_capacity/2.7
max_discharge = max_charge
layers = 3
neurons = 200
train_test_split = 0.6

past_features = ['solar_energy']
future_features = ['direct_rad', 'hour_sin','hour_cos']
opt_features = ['load', 'load_fcst','offtake','injection']

opt_future_features = future_features + opt_features
# Create domain_min and domain_max for the opt models
domain_min = [min(nl_data['solar_energy'][:676*24+24]), # past: solar energy
              min(nl_data['direct_rad'][:676*24+24]),   # future: direct radiation
              -1,-1,                                    # future: hour sin + cos
              0,0,0,0,                                  # future: optimization parameters
              0]                                        # target: solar energy

domain_max = [max(nl_data['solar_energy'][:676*24+24]), # past: solar energy
              max(nl_data['direct_rad'][:676*24+24]),   # future: direct radiation
              1,1,                                      # future: hour sin + cos
              1,1,1,1,                                  # future: optimization parameters
              1]                                        # target: solar energy

In [33]:
pvb_system = pvb.PV_battery(nl_data, building, battery_capacity, max_charge, max_discharge, self_consumption=False)

In [22]:
if full_mpc is False:
    dictionary_perfect = pvb_system.execute_optimization(time,23,'Perfect',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]
    dictionary_naive = pvb_system.execute_optimization(time,23,'Naive',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]
    dictionary_lstm = pvb_system.execute_optimization(time,23,'LSTM',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]
    dictionary_cvx = pvb_system.execute_optimization(time,23,'CVX',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]
    dictionary_lstm_cvx = pvb_system.execute_optimization(time,23,'LSTM_CVX',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]

    all_models = {'perfect': dictionary_perfect,
                  'naive': dictionary_naive,
                  'lstm': dictionary_lstm,
                  'cvx': dictionary_cvx,
                  'lstm_cvx': dictionary_lstm_cvx}

else:

    dictionary_list_perfect = pvb_system.execute_optimization(time,0,'Perfect',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)
    dictionary_list_naive = pvb_system.execute_optimization(time,0,'Naive',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)
    dictionary_list_lstm = pvb_system.execute_optimization(time,0,'LSTM',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)
    dictionary_list_cvx = pvb_system.execute_optimization(time,0,'CVX',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)
    dictionary_list_lstm_cvx = pvb_system.execute_optimization(time,0,'LSTM_CVX',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)

    all_models = {'perfect': dictionary_list_perfect,
                  'naive': dictionary_list_naive,
                  'lstm': dictionary_list_lstm,
                  'cvx': dictionary_list_cvx,
                  'lstm_cvx': dictionary_list_lstm_cvx}

IndexError: list index out of range

In [67]:
if full_mpc is False:
    with open('../results/optimisation/single_opt_building_' + str(building) + '_' + str(noise) + '_noise.pkl', "wb") as f:
        pickle.dump(all_models, f)
else:
    with open('../results/optimisation/full_mpc_building_' + str(building) + '_' + str(noise) + '_noise.pkl', "wb") as f:
        pickle.dump(all_models, f)